### XGBoost:
1) запуск алгоритма с параметрами по умолчанию и вывод некоторой статистики
2) запуск optuna-оптимизации по части гиперпараметров
3) визуализация optuna: важность параметров и контуры
4) запуск алгоритма с найденными гиперпараметрами и вывод предварительной статистики
5) сохраняем результаты дефолтного и оптимизированного алгоритма

In [1]:
from private.utils import get_reduced_mnist_data, memory_check
from public.classification_utils import XGB
from public.models import ClassificationProcessor

with memory_check():
    df = get_reduced_mnist_data()
    processor = ClassificationProcessor(df, "label")   
    processor.calculate({
        XGB : {},
    })
    processor.report(XGB)
    processor.pick_model(XGB)

extreme_gradient_boosting took 237.806 seconds

	extreme_gradient_boosting


pr_auc,roc_auc,accuracy
0.996639,0.999514,0.975786


,precision,recall,f1-score,support
0,0.974,0.994,0.984,1381.000
1,0.984,0.987,0.985,1575.000
2,0.974,0.974,0.974,1398.000
3,0.977,0.971,0.974,1428.000
4,0.978,0.969,0.974,1365.000
5,0.982,0.970,0.976,1263.000
6,0.980,0.984,0.982,1375.000
7,0.979,0.975,0.977,1459.000
8,0.972,0.972,0.972,1365.000
9,0.958,0.960,0.959,1391.000


Memory Increased by: 274.81 MB


#### запуск optuna-оптимизации по части гиперпараметров

In [2]:
from public.optuna_utils import OPT_XGB, optimize

with memory_check():
    study = optimize(
        model_type=OPT_XGB, 
        df=df, 
        target_column="label",
        n_trials=10
    )
    print(f"Наилучшие значения гиперпараметров {study.best_params}")
    print(f"pr_auc на обучающем наборе: {study.best_value:.4f}")

[I 2026-08-18 14:01:08,305] A new study created in memory with name: Extreme Gradient Boosting


  0%|          | 0/10 [00:00<?, ?it/s]

optuna_optimize took 4800.378 seconds
[I 2026-08-18 15:21:08,952] Trial 7 finished with value: 0.987849348742506 and parameters: {'n_estimators': 70, 'max_depth': 3, 'learning_rate': 0.27374627747116714}. Best is trial 7 with value: 0.987849348742506.
optuna_optimize took 5442.568 seconds
[I 2026-08-18 15:31:51,004] Trial 5 finished with value: 0.9949197432854758 and parameters: {'n_estimators': 55, 'max_depth': 5, 'learning_rate': 0.3266490467458776}. Best is trial 5 with value: 0.9949197432854758.
optuna_optimize took 5466.047 seconds
[I 2026-08-18 15:32:14,383] Trial 2 finished with value: 0.9920065541176374 and parameters: {'n_estimators': 64, 'max_depth': 4, 'learning_rate': 0.24608330516873114}. Best is trial 5 with value: 0.9949197432854758.
optuna_optimize took 5602.699 seconds
[I 2026-08-18 15:34:31,053] Trial 6 finished with value: 0.9794162463095273 and parameters: {'n_estimators': 89, 'max_depth': 3, 'learning_rate': 0.11339326229703688}. Best is trial 5 with value: 0.99491

#### Визуализация optuna:
1) Сравнение важности гиперпараметров
2) Отрисовка контура оптимизации. Помогает выбрать направление дальнейшей оптимизации в сторону "темных" областей

In [3]:
from optuna.visualization import plot_param_importances

plot_param_importances(study)

In [4]:
from optuna.visualization import plot_contour

plot_contour(study)

#### Применение найденных лучших гиперпараметров:

In [5]:
with memory_check():
    alter_title = f'{XGB}_tuned' 
    processor.calculate({
        XGB : {
            "n_estimators": 68,
            "max_depth": 5,
            "learning_rate": 0.42890132363208927,
            'alter_title': alter_title
        },
    })
    processor.report(alter_title)
    processor.pick_model(alter_title)

extreme_gradient_boosting took 130.894 seconds

	extreme_gradient_boosting_tuned


pr_auc,roc_auc,accuracy
0.996029,0.999424,0.974714


,precision,recall,f1-score,support
0,0.985,0.994,0.990,1381.000
1,0.984,0.986,0.985,1575.000
2,0.980,0.972,0.976,1398.000
3,0.973,0.970,0.971,1428.000
4,0.977,0.962,0.969,1365.000
5,0.981,0.968,0.974,1263.000
6,0.980,0.983,0.981,1375.000
7,0.973,0.978,0.976,1459.000
8,0.967,0.974,0.971,1365.000
9,0.947,0.958,0.953,1391.000


Memory Increased by: 654.35 MB


#### Мини-репорт:

In [6]:
dec_tr = next((model for model in processor.models if model.title == XGB), None)
dec_tr_tuned = next((model for model in processor.models if model.title == alter_title), None)

print(f"{XGB} : {alter_title} >> {dec_tr.pr_auc} : {dec_tr_tuned.pr_auc}")

extreme_gradient_boosting : extreme_gradient_boosting_tuned >> 0.9966386420015247 : 0.9960292482371609
